# EJERCICIO FINAL PYSPARK (DOCKER + KAFKA + PYSPARK)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Creamos la sesión de spark

In [ ]:
spark = (
    SparkSession.builder
        .appName("ProyectoIoT")
        .getOrCreate()
)
spark

spark.readStream
Indica que vamos a leer datos en modo streaming, es decir, datos que llegan continuamente.

.format("kafka")
Le decimos a Spark que la fuente de datos de streaming será Kafka.

.option("kafka.bootstrap.servers", "kafka:9092")
Especifica la dirección del clúster Kafka al que conectarse.
En este caso "kafka" es el nombre del contenedor Docker y 9092 es el puerto del broker.

.option("subscribe", "sensores")
Indica el topic de Kafka del que queremos leer mensajes → "sensores".

.option("startingOffsets", "latest")
Define desde qué punto empezar a leer:

"latest" → solo recibe mensajes nuevos, enviados después de iniciar el streaming.

"earliest" sería para leer todos los mensajes antiguos también.

.load()
Ejecuta la configuración y crea un DataFrame de streaming, donde cada fila representa un mensaje recibido desde Kafka.

In [ ]:
raw_df = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", "kafka:9092")  # <-- nombre del contenedor
         .option("subscribe", "sensores")
         .option("startingOffsets", "latest")
         .load()
)


Este bloque transforma los mensajes de Kafka en un formato que Spark puede procesar:

Define la estructura de los datos que esperamos recibir (sensor, valor, temperatura, humedad, estado, timestamp y UUID).

Convierte el JSON recibido en columnas individuales para poder trabajar con cada campo.

Crea una columna de tiempo (event_time) a partir del timestamp original para poder usarlo en ventanas y agregaciones temporales.

In [ ]:
schema = StructType([
    StructField("sensor_id", StringType()),
    StructField("value", DoubleType()),
    StructField("temperature", DoubleType()),
    StructField("humidity", DoubleType()),
    StructField("status", StringType()),
    StructField("timestamp", DoubleType()),
    StructField("uuid", StringType())
])

df = raw_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

df = df.withColumn("event_time", col("timestamp").cast("timestamp"))


Este bloque realiza agregaciones por ventana de tiempo sobre el DataFrame de streaming:

Define un watermark de 1 minuto para indicar a Spark hasta qué punto los datos antiguos se pueden considerar válidos. Esto ayuda a manejar retrasos y datos tardíos.

Agrupa los datos cada 30 segundos por sensor (sensor_id).

Calcula estadísticas dentro de cada ventana: promedio de valor, temperatura y humedad, y cuenta el número de eventos recibidos.

El resultado es un DataFrame de streaming con resúmenes temporales por sensor listo para análisis o escritura.

In [ ]:
agg_df = (
    df
    .withWatermark("event_time", "1 minute")   # ← IMPORTANTE
    .groupBy(
        window(col("event_time"), "30 seconds"),
        col("sensor_id")
    )
    .agg(
        avg("value").alias("avg_value"),
        avg("temperature").alias("avg_temp"),
        avg("humidity").alias("avg_humidity"),
        count("*").alias("num_events")
    )
)


Este bloque escribe los resultados del streaming en archivos Parquet de manera continua:

Define la carpeta de salida donde se guardarán los archivos Parquet (resultados/).

Usa un checkpoint (chk/) para que Spark recuerde qué datos ya procesó y pueda reiniciar de forma segura en caso de fallo.

Modo append: los nuevos datos se agregan continuamente a los archivos existentes.

Inicia el streaming, haciendo que las agregaciones se escriban en tiempo real mientras llegan nuevos datos.

El resultado es un conjunto de archivos Parquet que refleja las métricas agregadas por ventana y por sensor.

In [ ]:
parquet_query = (
    agg_df
    .writeStream
    .format("parquet")
    .option("path", "resultados/")
    .option("checkpointLocation", "chk/")
    .outputMode("append")
    .start()
)


A partir de aquí, te toca a ti, sigue las cuestiones planteadas en el Readme y completa el notebook. Despues guardatelo con los outputs de las celdas y súbelo al repo. Mucha suerte que ya lo teneis ;)

## 🧰 0. Preparación
Cargamos el DataFrame desde los archivos Parquet generados por el streaming y lo preparamos para el análisis.

In [ ]:
# Cargamos el DataFrame desde los archivos Parquet generados
df_resultados = spark.read.parquet("resultados/")

# Mostramos las columnas actuales
print("Columnas del DataFrame:")
print(df_resultados.columns)

### Ejercicio 1 - Exploración inicial
Realiza una exploración básica del DataFrame:
- Muestra el esquema completo
- Indica cuántas filas contiene
- ¿Cuántos sensores distintos (sensor_id) aparecen?

In [ ]:
# Mostrar el esquema completo del DataFrame
print("=== ESQUEMA DEL DATAFRAME ===")
df_resultados.printSchema()

In [ ]:
# Contar el número total de filas
num_filas = df_resultados.count()
print(f"=== NÚMERO TOTAL DE FILAS ===")
print(f"El DataFrame contiene {num_filas} filas")

In [ ]:
# Contar el número de sensores distintos
num_sensores = df_resultados.select("sensor_id").distinct().count()
print(f"=== SENSORES DISTINTOS ===")
print(f"Aparecen {num_sensores} sensores distintos")

# Mostrar los IDs de los sensores
print("\nLista de sensores:")
df_resultados.select("sensor_id").distinct().show()

### Ejercicio 2 - Transformaciones numéricas
Crea una columna basada en la diferencia o relación entre dos métricas numéricas del DataFrame.

Responde:
- ¿Cuál es el valor mínimo, máximo y medio de la nueva columna?

In [ ]:
# Crear una nueva columna: ratio entre temperatura y humedad (temp_humidity_ratio)
# También creamos la diferencia entre avg_temp y avg_humidity
df_transformado = df_resultados.withColumn(
    "temp_humidity_ratio", 
    F.col("avg_temp") / F.col("avg_humidity")
).withColumn(
    "temp_humidity_diff",
    F.col("avg_temp") - F.col("avg_humidity")
)

# Mostrar algunas filas con las nuevas columnas
print("=== DATAFRAME CON NUEVAS COLUMNAS ===")
df_transformado.select("sensor_id", "avg_temp", "avg_humidity", "temp_humidity_ratio", "temp_humidity_diff").show(10)

In [ ]:
# Calcular estadísticas de la nueva columna temp_humidity_ratio
print("=== ESTADÍSTICAS DE temp_humidity_ratio ===")
stats_ratio = df_transformado.agg(
    F.min("temp_humidity_ratio").alias("min_ratio"),
    F.max("temp_humidity_ratio").alias("max_ratio"),
    F.avg("temp_humidity_ratio").alias("avg_ratio")
)
stats_ratio.show()

# Calcular estadísticas de temp_humidity_diff
print("=== ESTADÍSTICAS DE temp_humidity_diff ===")
stats_diff = df_transformado.agg(
    F.min("temp_humidity_diff").alias("min_diff"),
    F.max("temp_humidity_diff").alias("max_diff"),
    F.avg("temp_humidity_diff").alias("avg_diff")
)
stats_diff.show()

### Ejercicio 3 - Filtrado avanzado
Aplica un filtro usando varias condiciones a la vez relacionadas con:
- Humedad
- Número de eventos
- Sensor

Responde:
- ¿Cuántos registros cumplen todas las condiciones aplicadas?

In [ ]:
# Aplicar filtro con múltiples condiciones:
# - Humedad promedio mayor a 40
# - Número de eventos mayor o igual a 2
# - Sensor S1 o S2

df_filtrado = df_resultados.filter(
    (F.col("avg_humidity") > 40) &
    (F.col("num_events") >= 2) &
    (F.col("sensor_id").isin(["S1", "S2"]))
)

# Contar registros que cumplen las condiciones
num_registros_filtrados = df_filtrado.count()

print("=== FILTRADO AVANZADO ===")
print(f"Condiciones aplicadas:")
print(f"  - avg_humidity > 40")
print(f"  - num_events >= 2")
print(f"  - sensor_id IN ('S1', 'S2')")
print(f"\nNúmero de registros que cumplen TODAS las condiciones: {num_registros_filtrados}")

In [ ]:
# Mostrar algunos de los registros filtrados
print("=== MUESTRA DE REGISTROS FILTRADOS ===")
df_filtrado.show(10)

### Ejercicio 4 - Agregaciones por sensor
Agrupa el DataFrame por sensor_id y obtén estadísticas descriptivas.

Responde:
- ¿Qué sensor tiene la mayor media en la variable analizada?
- ¿Qué sensor presenta la temperatura máxima?
- ¿Cuántos grupos o ventanas existen por sensor?

In [ ]:
# Agregar estadísticas por sensor
df_stats_sensor = df_resultados.groupBy("sensor_id").agg(
    F.avg("avg_value").alias("media_value"),
    F.avg("avg_temp").alias("media_temp"),
    F.avg("avg_humidity").alias("media_humidity"),
    F.max("avg_temp").alias("max_temp"),
    F.min("avg_temp").alias("min_temp"),
    F.sum("num_events").alias("total_eventos"),
    F.count("*").alias("num_ventanas")
).orderBy(F.desc("media_value"))

print("=== ESTADÍSTICAS POR SENSOR ===")
df_stats_sensor.show()

In [ ]:
# ¿Qué sensor tiene la mayor media en avg_value?
sensor_mayor_media = df_stats_sensor.first()
print(f"=== RESPUESTAS ===")
print(f"Sensor con mayor media de value: {sensor_mayor_media['sensor_id']} (media: {sensor_mayor_media['media_value']:.4f})")

In [ ]:
# ¿Qué sensor presenta la temperatura máxima?
sensor_max_temp = df_stats_sensor.orderBy(F.desc("max_temp")).first()
print(f"Sensor con temperatura máxima: {sensor_max_temp['sensor_id']} (max_temp: {sensor_max_temp['max_temp']:.4f})")

In [ ]:
# ¿Cuántos grupos o ventanas existen por sensor?
print("\n=== NÚMERO DE VENTANAS POR SENSOR ===")
df_stats_sensor.select("sensor_id", "num_ventanas").show()

### Ejercicio 5 - Ranking con funciones de ventana
Usa funciones de ventana para identificar, para cada sensor, el registro con el mayor valor de avg_value.

Responde:
- ¿Qué sensor obtiene el valor máximo global entre todos?

In [ ]:
# Definir la ventana particionada por sensor, ordenada por avg_value descendente
window_spec = Window.partitionBy("sensor_id").orderBy(F.desc("avg_value"))

# Añadir columna de ranking
df_con_ranking = df_resultados.withColumn("rank", F.row_number().over(window_spec))

# Filtrar solo el registro con mayor avg_value por sensor (rank = 1)
df_max_por_sensor = df_con_ranking.filter(F.col("rank") == 1)

print("=== REGISTRO CON MAYOR avg_value POR SENSOR ===")
df_max_por_sensor.select("sensor_id", "avg_value", "avg_temp", "avg_humidity", "window").show(truncate=False)

In [ ]:
# ¿Qué sensor obtiene el valor máximo global entre todos?
sensor_max_global = df_max_por_sensor.orderBy(F.desc("avg_value")).first()
print(f"=== SENSOR CON VALOR MÁXIMO GLOBAL ===")
print(f"Sensor: {sensor_max_global['sensor_id']}")
print(f"Valor máximo (avg_value): {sensor_max_global['avg_value']:.4f}")

### Ejercicio 6 - Join con tabla auxiliar
Crea un DataFrame auxiliar con información adicional sobre sensores (por ejemplo, categoría, ubicación o tipo) y realiza un join.

Responde:
- Muestra los cinco primeros registros del DataFrame ya unido.

In [ ]:
# Crear DataFrame auxiliar con información adicional de sensores
datos_sensores = [
    ("S1", "Temperatura", "Planta Baja", "Activo"),
    ("S2", "Humedad", "Primer Piso", "Activo"),
    ("S3", "Mixto", "Segundo Piso", "Mantenimiento"),
    ("S4", "Industrial", "Almacén", "Activo")
]

schema_aux = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("tipo", StringType(), True),
    StructField("ubicacion", StringType(), True),
    StructField("estado", StringType(), True)
])

df_info_sensores = spark.createDataFrame(datos_sensores, schema_aux)

print("=== DATAFRAME AUXILIAR DE SENSORES ===")
df_info_sensores.show()

In [ ]:
# Realizar el JOIN entre el DataFrame de resultados y la información auxiliar
df_joined = df_resultados.join(
    df_info_sensores,
    on="sensor_id",
    how="left"
)

print("=== 5 PRIMEROS REGISTROS DEL DATAFRAME UNIDO ===")
df_joined.show(5, truncate=False)

### Ejercicio 7 - Agrupación por ventana temporal
Extrae la marca de inicio de la ventana y agrúpala por unidades de tiempo truncadas (p. ej., minuto u hora).

Responde:
- ¿Cuántas ventanas hay por unidad temporal?
- ¿Cuál es la humedad media por cada unidad?

In [ ]:
# Extraer el inicio de la ventana y truncar por minuto
df_con_minuto = df_resultados.withColumn(
    "window_start", F.col("window.start")
).withColumn(
    "minuto", F.date_trunc("minute", F.col("window_start"))
)

print("=== DATAFRAME CON MARCA DE MINUTO ===")
df_con_minuto.select("sensor_id", "window", "window_start", "minuto").show(5, truncate=False)

In [ ]:
# Agrupar por minuto y calcular estadísticas
df_por_minuto = df_con_minuto.groupBy("minuto").agg(
    F.count("*").alias("num_ventanas"),
    F.avg("avg_humidity").alias("humedad_media"),
    F.avg("avg_temp").alias("temp_media"),
    F.avg("avg_value").alias("value_medio")
).orderBy("minuto")

print("=== ESTADÍSTICAS POR MINUTO ===")
print("¿Cuántas ventanas hay por minuto? y ¿Cuál es la humedad media por minuto?")
df_por_minuto.show(truncate=False)

### Ejercicio 8 - Repartición y particiones
Reparte el DataFrame por sensor_id y añade una columna indicando el identificador de partición.

Responde:
- ¿Cuál es la distribución de registros entre las particiones?

In [ ]:
# Reparticionar por sensor_id (4 particiones, una por sensor)
df_reparticionado = df_resultados.repartition(4, "sensor_id")

# Añadir columna con el identificador de partición
df_con_particion = df_reparticionado.withColumn(
    "partition_id", F.spark_partition_id()
)

print("=== DATAFRAME CON ID DE PARTICIÓN ===")
df_con_particion.select("sensor_id", "avg_value", "partition_id").show(10)

In [ ]:
# Distribución de registros entre particiones
print("=== DISTRIBUCIÓN DE REGISTROS POR PARTICIÓN ===")
df_distribucion = df_con_particion.groupBy("partition_id").agg(
    F.count("*").alias("num_registros"),
    F.collect_set("sensor_id").alias("sensores_en_particion")
).orderBy("partition_id")

df_distribucion.show(truncate=False)

### Ejercicio 9 - Detección de anomalías
Define un criterio personalizado de "anomalía" basado en los valores medios del DataFrame.

Responde:
- ¿Cuántos registros son anómalos?
- ¿Qué sensor tiene más anomalías?

In [ ]:
# Calcular media y desviación estándar de avg_value para definir anomalías
stats = df_resultados.agg(
    F.avg("avg_value").alias("mean_value"),
    F.stddev("avg_value").alias("std_value"),
    F.avg("avg_temp").alias("mean_temp"),
    F.stddev("avg_temp").alias("std_temp")
).collect()[0]

mean_value = stats["mean_value"]
std_value = stats["std_value"]
mean_temp = stats["mean_temp"]
std_temp = stats["std_temp"]

print(f"=== ESTADÍSTICAS PARA DETECCIÓN DE ANOMALÍAS ===")
print(f"Media de avg_value: {mean_value:.4f}")
print(f"Desviación estándar de avg_value: {std_value:.4f}")
print(f"Media de avg_temp: {mean_temp:.4f}")
print(f"Desviación estándar de avg_temp: {std_temp:.4f}")

In [ ]:
# Definir anomalía: valor fuera de 2 desviaciones estándar de la media
# o temperatura extrema (fuera de 1.5 std)
df_con_anomalias = df_resultados.withColumn(
    "es_anomalia",
    F.when(
        (F.abs(F.col("avg_value") - mean_value) > 2 * std_value) |
        (F.abs(F.col("avg_temp") - mean_temp) > 1.5 * std_temp),
        True
    ).otherwise(False)
)

# Contar registros anómalos
num_anomalias = df_con_anomalias.filter(F.col("es_anomalia") == True).count()
total_registros = df_con_anomalias.count()

print(f"=== DETECCIÓN DE ANOMALÍAS ===")
print(f"Criterio: |avg_value - media| > 2*std OR |avg_temp - media| > 1.5*std")
print(f"\nTotal de registros: {total_registros}")
print(f"Registros anómalos: {num_anomalias}")
print(f"Porcentaje de anomalías: {(num_anomalias/total_registros)*100:.2f}%")

In [ ]:
# ¿Qué sensor tiene más anomalías?
df_anomalias_sensor = df_con_anomalias.filter(F.col("es_anomalia") == True).groupBy("sensor_id").agg(
    F.count("*").alias("num_anomalias")
).orderBy(F.desc("num_anomalias"))

print("=== ANOMALÍAS POR SENSOR ===")
df_anomalias_sensor.show()

sensor_mas_anomalias = df_anomalias_sensor.first()
if sensor_mas_anomalias:
    print(f"\nSensor con más anomalías: {sensor_mas_anomalias['sensor_id']} ({sensor_mas_anomalias['num_anomalias']} anomalías)")

### Ejercicio 10 - Cruce de métricas
Crea una puntuación combinando varias columnas del DataFrame (por ejemplo, ponderando avg_value, temperatura y humedad).

Responde:
- ¿Qué sensor obtiene la puntuación más alta?
- ¿Cuál es el valor de dicha puntuación?

In [ ]:
# Crear puntuación ponderada:
# score = 0.5 * avg_value + 0.3 * avg_temp + 0.2 * avg_humidity
# Normalizamos primero para que las escalas sean comparables

# Obtener min/max para normalización
min_max = df_resultados.agg(
    F.min("avg_value").alias("min_value"), F.max("avg_value").alias("max_value"),
    F.min("avg_temp").alias("min_temp"), F.max("avg_temp").alias("max_temp"),
    F.min("avg_humidity").alias("min_hum"), F.max("avg_humidity").alias("max_hum")
).collect()[0]

# Crear columnas normalizadas y puntuación
df_con_score = df_resultados.withColumn(
    "norm_value", 
    (F.col("avg_value") - min_max["min_value"]) / (min_max["max_value"] - min_max["min_value"])
).withColumn(
    "norm_temp",
    (F.col("avg_temp") - min_max["min_temp"]) / (min_max["max_temp"] - min_max["min_temp"])
).withColumn(
    "norm_humidity",
    (F.col("avg_humidity") - min_max["min_hum"]) / (min_max["max_hum"] - min_max["min_hum"])
).withColumn(
    "score",
    0.5 * F.col("norm_value") + 0.3 * F.col("norm_temp") + 0.2 * F.col("norm_humidity")
)

print("=== DATAFRAME CON PUNTUACIÓN ===")
print("Score = 0.5 * norm_value + 0.3 * norm_temp + 0.2 * norm_humidity")
df_con_score.select("sensor_id", "avg_value", "avg_temp", "avg_humidity", "score").show(10)

In [ ]:
# Puntuación media por sensor
df_score_sensor = df_con_score.groupBy("sensor_id").agg(
    F.avg("score").alias("score_medio"),
    F.max("score").alias("score_max")
).orderBy(F.desc("score_medio"))

print("=== PUNTUACIÓN POR SENSOR ===")
df_score_sensor.show()

# Sensor con mayor puntuación media
mejor_sensor = df_score_sensor.first()
print(f"\n=== RESPUESTA ===")
print(f"Sensor con puntuación más alta (media): {mejor_sensor['sensor_id']}")
print(f"Valor de la puntuación media: {mejor_sensor['score_medio']:.4f}")
print(f"Puntuación máxima alcanzada: {mejor_sensor['score_max']:.4f}")

### Ejercicio 11 - Graficar DF
Crea dos gráficos, convirtiendo el DF de spark a pandas y utiliza la librería matplotlib para generar dos gráficos sobre alguna de las variables.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convertir DataFrame de Spark a Pandas
df_pandas = df_resultados.toPandas()

print(f"DataFrame convertido a Pandas: {len(df_pandas)} registros")
df_pandas.head()

In [ ]:
# Crear figura con dos subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Boxplot de avg_value por sensor
ax1 = axes[0]
df_pandas.boxplot(column='avg_value', by='sensor_id', ax=ax1)
ax1.set_title('Distribución de avg_value por Sensor')
ax1.set_xlabel('Sensor ID')
ax1.set_ylabel('Valor Promedio')
plt.suptitle('')  # Eliminar título automático del boxplot

# Gráfico 2: Scatter plot de temperatura vs humedad
ax2 = axes[1]
colors = {'S1': 'red', 'S2': 'blue', 'S3': 'green', 'S4': 'orange'}
for sensor in df_pandas['sensor_id'].unique():
    subset = df_pandas[df_pandas['sensor_id'] == sensor]
    ax2.scatter(subset['avg_temp'], subset['avg_humidity'], 
                label=sensor, alpha=0.6, c=colors.get(sensor, 'gray'))

ax2.set_title('Temperatura vs Humedad por Sensor')
ax2.set_xlabel('Temperatura Promedio')
ax2.set_ylabel('Humedad Promedio')
ax2.legend(title='Sensor')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('graficos_sensores.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Gráficos guardados como 'graficos_sensores.png'")

In [ ]:
# Gráfico adicional: Evolución temporal de las métricas
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Extraer start de la ventana para el eje temporal
df_pandas['window_start'] = df_pandas['window'].apply(lambda x: x['start'])
df_pandas_sorted = df_pandas.sort_values('window_start')

# Gráfico 1: Número de eventos por ventana temporal
ax1 = axes[0]
for sensor in df_pandas_sorted['sensor_id'].unique():
    subset = df_pandas_sorted[df_pandas_sorted['sensor_id'] == sensor]
    ax1.plot(subset['window_start'], subset['num_events'], 
             marker='o', label=sensor, alpha=0.7, markersize=4)

ax1.set_title('Número de Eventos por Ventana Temporal')
ax1.set_xlabel('Tiempo')
ax1.set_ylabel('Número de Eventos')
ax1.legend(title='Sensor')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Gráfico 2: Histograma de avg_value
ax2 = axes[1]
for sensor in df_pandas['sensor_id'].unique():
    subset = df_pandas[df_pandas['sensor_id'] == sensor]
    ax2.hist(subset['avg_value'], bins=20, alpha=0.5, label=sensor)

ax2.set_title('Distribución de avg_value por Sensor')
ax2.set_xlabel('Valor Promedio')
ax2.set_ylabel('Frecuencia')
ax2.legend(title='Sensor')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('graficos_temporales.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Gráficos temporales guardados como 'graficos_temporales.png'")